# 08 — MLflow Model Registry

Cierre del ciclo MLOps: versionar el modelo campeón, promoverlo por stages de despliegue (`staging` → `production`) y servirlo desde el registry en lugar de una ruta de checkpoint fija.

Usamos **aliases** de MLflow (`staging`, `production`) — el reemplazo moderno de las stage transitions deprecadas. La API carga el modelo con `MODEL_REGISTRY_ALIAS=production`.

**Flujo:**
1. Entrenar (o tener) un checkpoint `.pt`
2. `register()` → nueva versión en el registry con métricas
3. `promote(version, 'staging')` → validación
4. `promote(version, 'production')` → despliegue
5. `load_model(alias='production')` → servir

In [ ]:
import sys
from pathlib import Path

ROOT = Path().resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import warnings
warnings.filterwarnings('ignore')

import torch
import pandas as pd

from src.models.classifier import DocumentClassifier
from src.models.registry import ModelRegistry

## 1. Preparar un checkpoint

Usamos el checkpoint entrenado si existe; si no, generamos uno sin entrenar para demostrar el flujo del registry.

In [ ]:
ckpt_path = ROOT / 'models' / 'saved' / 'efficientnet_b0_best.pt'

if not ckpt_path.exists():
    print('No hay checkpoint entrenado; generando uno de demostración…')
    model = DocumentClassifier(pretrained=False)
    model.save(ckpt_path, metadata={'note': 'demo checkpoint (sin entrenar)'})

print(f'Checkpoint: {ckpt_path}  ({ckpt_path.stat().st_size / 1e6:.1f} MB)')

## 2. Registrar una versión

El registry usa el backend SQLite del proyecto (`mlflow.db`).

In [ ]:
registry = ModelRegistry(
    model_name='document-authenticator',
    tracking_uri=f"sqlite:///{ROOT / 'mlflow.db'}",
)

version = registry.register(
    ckpt_path,
    metrics={'val_f1': 0.94, 'val_auc': 0.97, 'val_accuracy': 0.93},
    params={'backbone': 'efficientnet_b0', 'input_size': 224},
    description='EfficientNet-B0, fine-tune en dos fases',
)
print(f'Registrada versión {version}')

## 3. Promover por stages

Primero a `staging` (validación), luego a `production` (despliegue).

In [ ]:
registry.promote(version, alias='staging')
print(f'v{version} → staging')

# Tras validar en staging, promover a production
registry.promote(version, alias='production')
print(f'v{version} → production')

assert registry.get_version_by_alias('production') == version

## 4. Inventario de versiones

In [ ]:
versions = registry.list_versions()
rows = []
for v in versions:
    rows.append({
        'version': v['version'],
        'aliases': ', '.join(v['aliases']) or '—',
        'val_f1': v['metrics'].get('val_f1', float('nan')),
        'val_auc': v['metrics'].get('val_auc', float('nan')),
        'description': v['description'],
    })
pd.DataFrame(rows).set_index('version')

## 5. Cargar el modelo de producción

Así es como la API resuelve el modelo cuando `MODEL_REGISTRY_ALIAS=production`.

In [ ]:
model = registry.load_model(alias='production', device='cpu')
model.eval()

dummy = torch.zeros(1, 3, 224, 224)
with torch.no_grad():
    prob = model.predict_proba(dummy).item()
print(f'Modelo de producción cargado · P(falsificado) sobre input dummy = {prob:.4f}')

## 6. Despliegue

Para que la API sirva desde el registry, basta con variables de entorno:

```bash
export MLFLOW_TRACKING_URI=sqlite:///mlflow.db
export MODEL_REGISTRY_NAME=document-authenticator
export MODEL_REGISTRY_ALIAS=production
uvicorn api.main:app
```

Si el registry no está disponible, la API hace fallback automático al checkpoint local (`MODEL_CHECKPOINT`) — no se cae el servicio.

**Rollback**: para volver a una versión anterior, basta con `registry.promote(version_anterior, 'production')`. El alias se reasigna atómicamente y la API la toma en el siguiente arranque.

Explora el registry completo con:
```bash
mlflow ui --backend-store-uri sqlite:///mlflow.db
```